In [1]:
import torch
#from modelComp.FluidGPT_FM import FluidGPT_FM
from modelComp.FluidGPT_B import FluidGPT_B
#from modelComp.FluidGPT_FM_d3 import FluidGPT_FM
from modelComp.utils import ACT_MAPPER, SKIPBLOCK_MAPPER


In [ ]:
# step size for ode solver
step_size = 0.05

norm = cm.colors.Normalize(vmax=50, vmin=0)

batch_size = 50000  # batch size
eps_time = 1e-2
T = torch.linspace(0,1,10)  # sample times
T = T.to(device=device)

x_init = torch.randn((batch_size, 2), dtype=torch.float32, device=device)
solver = ODESolver(velocity_model=wrapped_vf)  # create an ODESolver class
sol = solver.sample(time_grid=T, x_init=x_init, method='midpoint', step_size=step_size, return_intermediates=True)  # sample from the model



In [6]:
target = torch.randn(176, 5, 2, 128, 128)  # Example target tensor
tf = torch.rand(target.size(0), device=target.device)
print(tf)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
t_grid = torch.tensor(
    [0.01, 0.04, 0.10, 0.25, 0.75, 0.90, 0.96, 0.99],
    device=device
)
tf = t_grid[torch.arange(target.size(0), device=device) % len(t_grid)]
t_expand = tf.view(-1, 1, 1, 1, 1).repeat(
    1, target.shape[1], target.shape[2], target.shape[3], target.shape[4]
)

print(tf)

tensor([0.2338, 0.3589, 0.0912, 0.9165, 0.6947, 0.0191, 0.9007, 0.5319, 0.2895,
        0.9189, 0.1208, 0.4542, 0.3962, 0.1642, 0.1446, 0.3642, 0.4922, 0.9181,
        0.5054, 0.8182, 0.1447, 0.9940, 0.8002, 0.5969, 0.3555, 0.2045, 0.3131,
        0.0927, 0.9297, 0.5773, 0.1308, 0.1877, 0.2347, 0.2658, 0.5640, 0.4099,
        0.8339, 0.3287, 0.2417, 0.0360, 0.8346, 0.6772, 0.0696, 0.5297, 0.9790,
        0.9391, 0.5124, 0.3654, 0.4493, 0.1840, 0.6323, 0.3440, 0.6309, 0.1587,
        0.0238, 0.1178, 0.3167, 0.2131, 0.5498, 0.5511, 0.4978, 0.2134, 0.6481,
        0.7595, 0.8279, 0.2018, 0.5112, 0.6977, 0.6953, 0.6843, 0.6207, 0.6000,
        0.5756, 0.8061, 0.7283, 0.1144, 0.5615, 0.7077, 0.1757, 0.3398, 0.8444,
        0.7561, 0.8505, 0.2145, 0.3045, 0.0528, 0.3432, 0.3618, 0.3031, 0.1590,
        0.1745, 0.9013, 0.8243, 0.2519, 0.0304, 0.8204, 0.8754, 0.9587, 0.3361,
        0.4218, 0.8266, 0.1692, 0.0855, 0.8083, 0.3109, 0.5464, 0.2500, 0.1720,
        0.9833, 0.5768, 0.4427, 0.5214, 

In [11]:
with torch.no_grad():
    steps = 20
    B = 12
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    eps = 1e-3
    dt = (1.0 - 2 * eps) / steps
    dt_sum = 0
    for k in range(steps):
        t = eps + k * dt
        t_batch = torch.full((B,), t, device=device)
        print(t)
        dt_sum += dt
        #pred = self(xt, t_batch)
        #xt = xt + dt * pred
    print("dt sum:", dt_sum)

0.001
0.0509
0.1008
0.1507
0.2006
0.2505
0.3004
0.3503
0.4002
0.4501
0.5
0.5498999999999999
0.5998
0.6497
0.6996
0.7494999999999999
0.7994
0.8493
0.8992
0.9490999999999999
dt sum: 0.9980000000000004


In [ ]:
#model = FluidGPT_B
model = FluidGPT_B(emb_dim=96,
                            data_dim=[1, 21, 2, 128, 128],
                            patch_size=(8, 8),
                            hiddenout_dim=512,
                            depth=2,
                            stage_depths=[12,12,8,12,12],
                            num_heads=[4,8,16,8,4],
                            window_size=4,
                            use_flex_attn=True,
                            act=ACT_MAPPER["gelu"],
                            skip_connect=SKIPBLOCK_MAPPER["convnext"],
                            gradient_flowthrough=[True, True, True],
                            )

x = torch.randn(1, 21, 2, 128, 128)
y = model(x)
print(y.shape)
param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(param)

torch.Size([1, 21, 2, 128, 128])
17733616


In [3]:
from modelComp.FluidGPT_FM import FluidGPT_FM
model = FluidGPT_FM(emb_dim=96,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(8, 8),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=2,
                        stage_depths=[12,12,8,12,12],
                        num_heads=[4,8,16,8,4],
                        window_size=4,
                        use_flex_attn=True,
                        causal_attn="custom",
                        act=ACT_MAPPER["gelu"],
                        skip_connect=SKIPBLOCK_MAPPER["convnext"],
                        gradient_flowthrough=[True, True, True],
                        enable_final_layer=False
                        )
# count param
param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(param)

53675136


In [2]:
from modelComp.FluidGPT_FM_d3 import FluidGPT_FM
model = FluidGPT_FM(emb_dim=48,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(4, 4),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=3,
                        stage_depths=[12,12,12,8,12, 12,12],
                        num_heads=[4,4,8,16,8,4,4],
                        window_size=4,
                        use_flex_attn=True,
                        causal_attn="custom",
                        act=ACT_MAPPER["gelu"],
                        skip_connect=SKIPBLOCK_MAPPER["convnext"],
                        gradient_flowthrough=[True, True, True],
                        enable_final_layer=False
                        )
# count param
param = sum(p.numel() for p in model.parameters() if p.requires_grad)
model = model.cuda()
print(param)
x = torch.randn(1,2,21,128,128)
x = x.cuda()
t = torch.Tensor([0.5])
t = t.cuda()
y = model(x,t)

print(y.shape)
param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(param)

/usr/local/lib/python3.11/dist-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


54968912
torch.Size([1, 2, 21, 128, 128])
54968912


In [ ]:
from modelComp.FluidGPT_FM_d3 import FluidGPT_FM
model = FluidGPT_FM(emb_dim=48,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(4, 4),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=3,
                        stage_depths=[12,12,12,8,12,12,12],
                        num_heads=[4,4,8,16,8,4,4],
                        window_size=4,
                        use_flex_attn=True,
                        causal_attn="custom",
                        act=ACT_MAPPER["gelu"],
                        skip_connect=SKIPBLOCK_MAPPER["convnext"],
                        gradient_flowthrough=[True, True, True],
                        enable_final_layer=False
                        )
# count param
param = sum(p.numel() for p in model.parameters() if p.requires_grad)
#model = model.cuda()
print(param)
x = torch.randn(1,2,21,128,128)
#x = x.cuda()
t = torch.Tensor([0.5])
#t = t.cuda()
y = model(x,t)

print(y.shape)
param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(param)

54968912
torch.Size([1, 2, 21, 128, 128])
54968912


# 8 models

In [3]:
import torch
#from modelComp.FluidGPT_FM import FluidGPT_FM
from modelComp.FluidGPT_B import FluidGPT_B
#from modelComp.FluidGPT_FM_d3 import FluidGPT_FM
from modelComp.utils import ACT_MAPPER, SKIPBLOCK_MAPPER
print(SKIPBLOCK_MAPPER.keys())
print(ACT_MAPPER.keys())
def params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
    

dict_keys(['convnext', 'resblock', 'none'])
dict_keys(['relu', 'gelu', 'swiglu', 'leaky_relu'])


In [3]:
from modelComp.FluidGPT_FM_d3 import FluidGPT_FM
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
model = FluidGPT_FM(emb_dim=64,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(4, 4),
                        hiddenout_dim=256, 
                        flowmatching_emb_dim=256,
                        depth=3,
                        stage_depths=[6,6,6,6,6,6,6],
                        num_heads=[4,8,16,32,16,8,4],
                        window_size=4,
                        use_flex_attn=True,
                        causal_attn="custom",
                        act=ACT_MAPPER["gelu"],
                        skip_connect=SKIPBLOCK_MAPPER["convnext"],
                        gradient_flowthrough=[True, True, True],
                        enable_final_layer=False
                        )
print(params(model))
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
y = model(x,t)
print(y.shape)
model = FluidGPT_FM(emb_dim=48,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(4, 4),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=3,
                        stage_depths=[12,12,12,10,12,12,12],
                        num_heads=[4,8,16,32,16,8,4],
                        window_size=4,
                        use_flex_attn=True,
                        causal_attn="custom",
                        act=ACT_MAPPER["gelu"],
                        skip_connect=SKIPBLOCK_MAPPER["convnext"],
                        gradient_flowthrough=[True, True, True],
                        enable_final_layer=False
                        )
print(params(model))
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
y = model(x,t)
print(y.shape)
model = FluidGPT_FM(emb_dim=48,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(4, 4),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=3,
                        stage_depths=[12,12,12,8,12,12,12],
                        num_heads=[4,8,16,32,16,8,4],
                        window_size=4,
                        use_flex_attn=True,
                        causal_attn="custom",
                        act=ACT_MAPPER["gelu"],
                        skip_connect=SKIPBLOCK_MAPPER["convnext"],
                        gradient_flowthrough=[True, True, False],
                        enable_final_layer=False
                        )
print(params(model))
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
y = model(x,t)
print(y.shape)
model = FluidGPT_FM(emb_dim=48,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(4, 4),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=3,
                        stage_depths=[12,12,12,8,12,12,12],
                        num_heads=[4,8,16,32,16,8,4],
                        window_size=4,
                        use_flex_attn=True,
                        causal_attn="custom",
                        act=ACT_MAPPER["gelu"],
                        skip_connect=SKIPBLOCK_MAPPER["convnext"],
                        gradient_flowthrough=[True, False, False],
                        enable_final_layer=False
                        )
print(params(model))
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
y = model(x,t)
print(y.shape)
model = FluidGPT_FM(emb_dim=48,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(4, 4),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=3,
                        stage_depths=[12,12,12,8,12,12,12],
                        num_heads=[4,8,16,32,16,8,4],
                        window_size=4,
                        use_flex_attn=True,
                        causal_attn="custom",
                        act=ACT_MAPPER["gelu"],
                        skip_connect=SKIPBLOCK_MAPPER["none"],
                        gradient_flowthrough=[True, True, True],
                        enable_final_layer=False
                        )
print(params(model))
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
y = model(x,t)
print(y.shape)
model = FluidGPT_FM(emb_dim=48,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(4, 4),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=3,
                        stage_depths=[12,12,12,8,12,12,12],
                        num_heads=[4,8,16,32,16,8,4],
                        window_size=4,
                        use_flex_attn=False,
                        causal_attn="custom",
                        act=ACT_MAPPER["relu"],
                        skip_connect=SKIPBLOCK_MAPPER["convnext"],
                        gradient_flowthrough=[True, True, True],
                        enable_final_layer=False
                        )
print(params(model))
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
y = model(x,t)
print(y.shape)
model = FluidGPT_FM(emb_dim=48,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(4, 4),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=3,
                        stage_depths=[12,12,12,8,12,12,12],
                        num_heads=[4,8,16,32,16,8,4],
                        window_size=4,
                        use_flex_attn=True,
                        causal_attn="custom",
                        act=ACT_MAPPER["relu"],
                        skip_connect=SKIPBLOCK_MAPPER["none"],
                        gradient_flowthrough=[True, True, True],
                        enable_final_layer=False
                        )
print(params(model))
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
y = model(x,t)
print(y.shape)
model = FluidGPT_FM(emb_dim=48,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(4, 4),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=3,
                        stage_depths=[12,12,12,8,12,12,12],
                        num_heads=[4,8,16,32,16,8,4],
                        window_size=4,
                        use_flex_attn=True,
                        causal_attn="custom",
                        act=ACT_MAPPER["relu"],
                        skip_connect=SKIPBLOCK_MAPPER["convnext"],
                        gradient_flowthrough=[True, True, True],
                        enable_final_layer=True
                        )
print(params(model))
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
y = model(x,t)
print(y.shape)


/home/tpharmsen/miniconda3/envs/fluidGPT311/lib/python3.11/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


55816720
torch.Size([1, 2, 21, 128, 128])
61423664
torch.Size([1, 2, 21, 128, 128])
55100400
torch.Size([1, 2, 21, 128, 128])
55100400
torch.Size([1, 2, 21, 128, 128])
52339488
torch.Size([1, 2, 21, 128, 128])
55099472
torch.Size([1, 2, 21, 128, 128])
52339488
torch.Size([1, 2, 21, 128, 128])
55100510
torch.Size([1, 2, 21, 128, 128])


# 4 d2 models

In [5]:
from modelComp.FluidGPT_FM import FluidGPT_FM
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
model = FluidGPT_FM(emb_dim=128,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(8, 8),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=2,
                        stage_depths=[6,6,6,6,6],
                        num_heads=[4,8,16,8,4],
                        window_size=4,
                        use_flex_attn=False,
                        causal_attn="custom",
                        act=ACT_MAPPER["gelu"],
                        skip_connect=SKIPBLOCK_MAPPER["none"],
                        gradient_flowthrough=[True, True, True],
                        enable_final_layer=False
                        )
print(params(model))
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
y = model(x,t)
print(y.shape)
model = FluidGPT_FM(emb_dim=128,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(8, 8),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=2,
                        stage_depths=[6,6,6,6,6],
                        num_heads=[4,8,16,8,4],
                        window_size=4,
                        use_flex_attn=False,
                        causal_attn="custom",
                        act=ACT_MAPPER["relu"],
                        skip_connect=SKIPBLOCK_MAPPER["none"],
                        gradient_flowthrough=[True, True, True],
                        enable_final_layer=False
                        )
print(params(model))
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
y = model(x,t)
print(y.shape)
model = FluidGPT_FM(emb_dim=96,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(8, 8),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=2,
                        stage_depths=[9,9,8,9,9],
                        num_heads=[4,8,16,8,4],
                        window_size=4,
                        use_flex_attn=False,
                        causal_attn="custom",
                        act=ACT_MAPPER["gelu"],
                        skip_connect=SKIPBLOCK_MAPPER["none"],
                        gradient_flowthrough=[True, True, True],
                        enable_final_layer=False
                        )
print(params(model))
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
y = model(x,t)
print(y.shape)
model = FluidGPT_FM(emb_dim=96,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(8, 8),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=2,
                        stage_depths=[9,9,8,9,9],
                        num_heads=[4,8,16,8,4],
                        window_size=4,
                        use_flex_attn=False,
                        causal_attn="custom",
                        act=ACT_MAPPER["relu"],
                        skip_connect=SKIPBLOCK_MAPPER["none"],
                        gradient_flowthrough=[True, True, True],
                        enable_final_layer=False
                        )
print(params(model))
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
y = model(x,t)
print(y.shape)
model = FluidGPT_FM(emb_dim=128,
                        data_dim=[1, 21, 2, 128, 128],
                        embedder_type="linear",
                        patch_size=(4, 4),
                        hiddenout_dim=512, 
                        flowmatching_emb_dim=512,
                        depth=2,
                        stage_depths=[6,6,6,6,6],
                        num_heads=[4,8,16,8,4],
                        window_size=4,
                        use_flex_attn=False,
                        causal_attn="custom",
                        act=ACT_MAPPER["relu"],
                        skip_enable=False,
                        skip_connect=SKIPBLOCK_MAPPER["none"],
                        gradient_flowthrough=[True, True, True],
                        enable_final_layer=False
                        )
print(params(model))
x = torch.randn(1,2,21,128,128)
t = torch.Tensor([0.5])
y = model(x,t)
print(y.shape)


50686144
torch.Size([1, 2, 21, 128, 128])
50686144
torch.Size([1, 2, 21, 128, 128])
45080512
torch.Size([1, 2, 21, 128, 128])
45080512
torch.Size([1, 2, 21, 128, 128])
50589792
torch.Size([1, 2, 21, 128, 128])
